In [ ]:
import os 
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_retrieval_chain 
#Combines retrieved documents by injecting them directly into the prompt. 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# Connects the retriever and LLM into an end-to-end RAG pipeline.

# Overall, these imports support loading external knowledge, embedding and storing it,
# retrieving relevant context at query time, and generating grounded LLM responses. 

load_dotenv() 
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
model=ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# 1. Load , chunk and index the content of the blog to create a retriever 

# This code loads a blog article from a URL, extracts only the main content using BeautifulSoup
# (via specific HTML classes), and converts it into LangChain Document objects. The content is
# then split into overlapping text chunks (chunk_size=1000, overlap=100), embedded, stored in
# a Chroma vector database, and exposed as a retriever for semantic search in RAG pipelines.

import bs4 
loader=WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header") 
        )
    ),
)
docs=loader.load()

# Chunking 
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
splits=text_splitter.split_documents(docs)
vectorstore=Chroma.from_documents(documents=splits,embedding=embeddings)
retriever=vectorstore.as_retriever()

In [10]:
## 2. Prompt Template

system_prompt = ( 
    "you are an assistant for question-answering tasks."
    "use the following pieces of retrieved context to answer"
    "the question.if you dont know the answer,say that you"
    "dont know. Use three sentences maximum and keep the"
    "answer concise"
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
    ("system", system_prompt),
    ("human","{input}")
    ]
)

In [12]:
# Create_Stuff_documents_chain function will combine all the webscrapped document into single document and replace 
# {context} value 

question_answer=create_stuff_documents_chain(model,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer)
response=rag_chain.invoke({"input":"what is self reflection"})
response

{'input': 'what is self reflection',
 'context': [Document(id='82d965e8-6313-49d1-82d6-c2df09e900e3', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Self-Reflection#\nSelf-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.\nReAct (Yao et al. 2023) integrates reasoning and acting within LLM by extending the action space to be a combination of task-specific discrete actions and the language space. The former enables LLM to interact with the environment (e.g. use Wikipedia search API), while the latter prompting LLM to generate reasoning traces in natural language.\nThe ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:\nThought: ...\nAction: ...\nObservation: ...\n... (Repeated many times)'),
  Document(id='575b844c-b56c-4710-a5

### Adding history thru memory 

In [20]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder 

# System prompt 
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
    "\n\n"
    "{context}"
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever=create_history_aware_retriever(model,retriever,contextualize_q_prompt)
history_aware_retriever


RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000016E0BD2AB90>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(ta

In [19]:


qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain=create_stuff_documents_chain(model,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [21]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="What is Self-Reflection"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2="Tell me more about it?"
response2=rag_chain.invoke({"input":question,"chat_history":chat_history})
print(response2['answer'])

Self-reflection is a process that allows agents to improve by refining past action decisions and correcting previous mistakes through iterative learning.


### Second Method to retrieve history 

In [22]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [23]:
conversational_rag_chain.invoke(
    {"input": "What is Task Decomposition?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Task decomposition is the process of breaking down a complex task into smaller, more manageable subtasks or steps. This is often done to make the task easier to understand, plan, and execute.'

In [24]:
conversational_rag_chain.invoke(
    {"input": "What are common ways of doing it?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]

'Task decomposition can be done through simple prompting like "Steps for XYZ.\\n1.", using task-specific instructions, or with human inputs.'

In [ ]:
# This code wraps a RAG pipeline with session-based chat memory using RunnableWithMessageHistory. 
# It stores conversation history per session_id, injects past messages into both retrieval and answer prompts, 
# and ensures follow-up questions are rewritten into standalone queries for accurate document retrieval. 
# The result is a fully conversational, production-ready RAG system that maintains context across multiple user turns.